# SupportMind — Practical Fine-Tuning with Unsloth

This lab builds a small customer-support language model workflow using Qwen3-4B, Unsloth, LoRA/QLoRA, Hugging Face Datasets, and TRL. It covers dataset engineering, chat-template-aware tokenization, supervised fine-tuning, evaluation, adapter export, and final inference.

**Execution note:** model-weight downloads and GPU-intensive operations are disabled by default. The complete model-dependent code is included, but it runs only after you explicitly enable the relevant flags. Tokenizer/configuration inspection and model-independent validation can be performed without downloading the Qwen3-4B weights.


### 0. Environment Setup

This notebook uses Unsloth, PyTorch, Hugging Face Transformers, Datasets, and TRL. Model-weight downloads and GPU-intensive operations are disabled by default. You can enable them later by changing the run flags after reviewing the configuration and confirming that the runtime has enough GPU memory.


Inspect the available PyTorch/CUDA runtime and GPU capabilities. The results help verify that the selected FP16 training path is compatible with the current accelerator.


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print(f"Total VRAM: {gpu.total_memory / 1024**3:.2f} GiB")
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    print("FP16 supported: True")
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    print("CUDA is unavailable. Model training requires a CUDA GPU in this notebook.")


Install the Python packages required by the following workflow. Package installation is kept in notebook cells so the environment can be reproduced in a fresh Colab runtime.


In [ ]:
%pip install -q unsloth datasets scikit-learn matplotlib trl


Install the Python packages required by the following workflow. Package installation is kept in notebook cells so the environment can be reproduced in a fresh Colab runtime.


In [ ]:
# Optional: install the Unsloth kernels only if your runtime needs them.
# The notebook does not require xformers for its core workflow.
# %pip install -q --no-deps xformers


Import the core libraries, print their installed versions, and define execution flags. Expensive model downloads, training, evaluation, and export remain disabled by default so lightweight notebook sections can run independently.


In [ ]:
import unsloth
import torch
import transformers
import trl

print("Unsloth:", unsloth.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)

# Keep expensive model operations disabled until you explicitly choose to run them.
RUN_MODEL_DOWNLOADS = False
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_EXPORT = False

MODEL_READY = False
TRAINING_COMPLETED = False


### 1. Understanding Qwen3-4B

This section introduces the base model used in the lab. The values below are model-family specifications; the next configuration-inspection cell retrieves the values from the model repository so that the notebook does not rely only on hard-coded assumptions.

| Parameter | Description |
|---|---|
| Architecture | Dense decoder-only Transformer |
| Parameter scale | Approximately 4 billion parameters |
| Layers | Retrieved from the model configuration |
| Attention heads | Retrieved from the model configuration |
| KV heads | Retrieved from the model configuration |
| Context length | Retrieved from the model configuration |
| Vocabulary size | Retrieved from the tokenizer/configuration |
| License | Check the model card before redistribution |
| Original weight format | Check the model repository; runtime loading dtype is configured separately |


Inspect the available PyTorch/CUDA runtime and GPU capabilities. The results help verify that the selected FP16 training path is compatible with the current accelerator.


In [ ]:
import torch
import transformers
import unsloth

print("========== Environment ==========")

print(f"PyTorch Version: {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"Unsloth Version: {unsloth.__version__}")

print("\n========== GPU Information ==========")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)

    print(f"GPU: {gpu.name}")
    print(f"Total VRAM: {gpu.total_memory / 1024**3:.2f} GB")
    print(f"CUDA Capability: {gpu.major}.{gpu.minor}")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")
    print(f"FP16 Supported: True")
else:
    print("CUDA is not available!")

Set the Hugging Face model identifier and retrieve its configuration metadata. This downloads only the small configuration files, not the model weights.


In [ ]:
from transformers import AutoConfig, AutoTokenizer

model_name = "Qwen/Qwen3-4B"

print("Loading model configuration...")

config = AutoConfig.from_pretrained(model_name)

print("\n========== Model Configuration ==========")

print(f"Model Type: {config.model_type}")
print(f"Architecture: {config.architectures}")
print(f"Hidden Size: {config.hidden_size}")
print(f"Number of Layers: {config.num_hidden_layers}")
print(f"Attention Heads: {config.num_attention_heads}")
print(f"KV Heads: {config.num_key_value_heads}")
print(f"Vocabulary Size: {config.vocab_size}")
print(f"Maximum Position Embeddings: {config.max_position_embeddings}")

Load the tokenizer and inspect its special tokens and chat template. Tokenizer files are much smaller than model weights; this step prepares text formatting and tokenization without loading the 4-billion-parameter model.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("========== Tokenizer Information ==========")

print(f"Tokenizer Class: {tokenizer.__class__.__name__}")
print(f"Vocabulary Size: {len(tokenizer)}")
print(f"EOS Token: {tokenizer.eos_token}")
print(f"PAD Token: {tokenizer.pad_token}")

print("\n========== Chat Template ==========")

print(tokenizer.chat_template[:1500])

### 2. Baseline Evaluation

A baseline is the behavior of the untouched pretrained model before fine-tuning. Comparing it with the fine-tuned model helps distinguish improvements caused by training from capabilities the base model already had. Loading the full model requires downloading several gigabytes of weights, so the optional baseline cell is disabled by default.


Optionally load the untouched Qwen3-4B model in FP16 and generate one qualitative response before fine-tuning. This provides a baseline for later comparison, but it downloads the full model weights and is disabled by default.


In [ ]:
# Optional baseline loading and one qualitative generation before fine-tuning.
# This downloads the full FP16 model weights and is disabled by default.

RUN_BASELINE_MODEL = False
baseline_model = None

if RUN_BASELINE_MODEL:
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required for this baseline experiment.")

    from transformers import AutoModelForCausalLM

    baseline_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16,
        device_map="auto"
    )
    baseline_model.eval()

    baseline_messages = [
        {
            "role": "user",
            "content": "I received my shoes, but they are damaged. I want to return them."
        }
    ]

    baseline_prompt = tokenizer.apply_chat_template(
        baseline_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    baseline_inputs = tokenizer(
        baseline_prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(baseline_model.device)

    with torch.inference_mode():
        baseline_outputs = baseline_model.generate(
            **baseline_inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    baseline_new_tokens = baseline_outputs[0][baseline_inputs["input_ids"].shape[1]:]
    baseline_text = tokenizer.decode(
        baseline_new_tokens,
        skip_special_tokens=True
    )

    print("Baseline response:")
    print(baseline_text)

    print(f"Allocated VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")
    print(f"Reserved VRAM: {torch.cuda.memory_reserved() / 1024**3:.2f} GiB")
else:
    print("Baseline model loading and generation skipped.")


### 3. Dataset Engineering

This section defines SupportMind's output contract, creates seed and augmented customer-support examples, validates their structure, checks dataset balance and duplicates, and creates leakage-safe train, validation, and test splits. Augmented paraphrases share a `group_id` with their source example so related samples cannot leak across splits.


Define the allowed intent labels used by the dataset, model instructions, output validator, and evaluation metrics. Keeping one shared list prevents label drift between stages.


In [ ]:
INTENTS = [
    "order_status",
    "shipping_issue",
    "refund_request",
    "return_request",
    "product_information",
    "technical_issue",
    "billing_issue",
    "complaint",
    "other"
]

Declare the expected output fields and their conceptual types. This schema is reused as the contract for dataset validation and inference-time output checks.


In [ ]:
OUTPUT_SCHEMA = {
    "intent": "string",
    "response": "string",
    "needs_clarification": "boolean"
}

Define SupportMind's behavioral contract: supported intents, clarification policy, factuality constraints, and the exact JSON response schema expected from the assistant.


In [ ]:
SYSTEM_PROMPT = """
You are SupportMind, a professional customer support assistant.

Your responsibilities:
1. Identify the customer's primary intent.
2. Provide a helpful and professional response.
3. Ask for clarification when required.
4. Never invent information.
5. Focus on the customer's primary requested action
   when selecting an intent.

Supported intents:
- order_status
- shipping_issue
- refund_request
- return_request
- product_information
- technical_issue
- billing_issue
- complaint
- other

Return only a valid JSON object containing exactly:
- intent: string
- response: string
- needs_clarification: boolean

Do not include Markdown or explanations outside JSON.
"""

Create three small illustrative conversation records to demonstrate the required `messages` format before building the larger SupportMind dataset.


In [ ]:
import json

train_samples = [
    {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": "Where is my order? My order ID is 12345."
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "intent": "order_status",
                    "response": "I can help you check your order. Please allow me a moment to review its status.",
                    "needs_clarification": False
                })
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": "I want a refund."
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "intent": "refund_request",
                    "response": "I can help you with your refund request. Could you provide your order ID?",
                    "needs_clarification": True
                })
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": "I want to return my shoes because they arrived damaged."
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "intent": "return_request",
                    "response": "I'm sorry your shoes arrived damaged. Could you provide your order ID so I can help you with the return?",
                    "needs_clarification": True
                })
            }
        ]
    }
]

Validate each sample's message roles, JSON structure, supported intent, response type, and clarification flag. Early schema checks prevent malformed examples from silently entering training.


In [ ]:
def validate_sample(sample):
    messages = sample["messages"]

    if len(messages) != 3:
        raise ValueError("Each sample must contain system, user, and assistant messages.")

    expected_roles = ["system", "user", "assistant"]
    actual_roles = [message.get("role") for message in messages]

    if actual_roles != expected_roles:
        raise ValueError(f"Unexpected message roles: {actual_roles}")

    output = json.loads(messages[2]["content"])

    if not isinstance(output, dict):
        raise ValueError("Assistant output must be a JSON object.")

    required_fields = {
        "intent",
        "response",
        "needs_clarification"
    }

    if set(output) != required_fields:
        raise ValueError("Assistant output does not match the required schema.")

    if output["intent"] not in INTENTS:
        raise ValueError(f"Unsupported intent: {output['intent']}")

    if not isinstance(output["response"], str) or not output["response"].strip():
        raise ValueError("The response must be a non-empty string.")

    if not isinstance(output["needs_clarification"], bool):
        raise ValueError("needs_clarification must be a boolean.")

    return True


for sample in train_samples:
    validate_sample(sample)

print(f"Validated {len(train_samples)} example samples.")


Build one consistent conversational training record from a user message and its target annotation. The function serializes the assistant target as JSON and attaches the shared system prompt.


In [ ]:
def make_sample(
    user_message,
    intent,
    response,
    needs_clarification,
    group_id
):
    assistant_output = {
        "intent": intent,
        "response": response,
        "needs_clarification": needs_clarification
    }

    return {
        "group_id": group_id,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_message
            },
            {
                "role": "assistant",
                "content": json.dumps(
                    assistant_output,
                    ensure_ascii=False
                )
            }
        ]
    }

Create the seed dataset: representative customer messages paired with intent labels, target responses, clarification flags, and group identifiers. These examples establish the initial behavior the model will learn.


In [ ]:
seed_data = [

    # 1. ORDER STATUS

    (
        "Where is my order? My order ID is 12345.",
        "order_status",
        "I can help you with your order inquiry. I don't have direct access to live order information, but you can check the latest status using your order ID on our order tracking page.",
        False,
        "G001"
    ),

    (
        "Can you tell me where my package is?",
        "order_status",
        "I'd be happy to help you check your order. Could you provide your order ID so I can guide you through the tracking process?",
        True,
        "G002"
    ),

    (
        "I placed an order three days ago, but I haven't received any updates.",
        "order_status",
        "I understand your concern. Could you share your order ID so I can help you find the appropriate way to check its status?",
        True,
        "G003"
    ),


    # 2. SHIPPING ISSUE

    (
        "My package is five days late.",
        "shipping_issue",
        "I'm sorry your package is delayed. Could you provide your order ID so I can guide you through the available shipping support options?",
        True,
        "G004"
    ),

    (
        "The tracking information says delivered, but I never received my package.",
        "shipping_issue",
        "I'm sorry you haven't received your package. Could you share your order ID and confirm whether the tracking page shows a delivery time or proof of delivery?",
        True,
        "G005"
    ),

    (
        "My package arrived with damaged packaging.",
        "shipping_issue",
        "I'm sorry your package arrived in that condition. Could you provide your order ID and briefly describe the damage to the packaging?",
        True,
        "G006"
    ),


    # 3. REFUND REQUEST

    (
        "I want a refund for my purchase.",
        "refund_request",
        "I can help you understand the refund process. Could you provide your order ID so I can guide you through the next steps?",
        True,
        "G007"
    ),

    (
        "The product was defective and I want my money back.",
        "refund_request",
        "I'm sorry the product was defective. Could you provide your order ID and briefly describe the defect so I can guide you through the refund process?",
        True,
        "G008"
    ),

    (
        "I received a refund confirmation, but the money hasn't appeared in my account.",
        "refund_request",
        "I understand your concern. Could you share your order ID and the date of the refund confirmation? I can help you understand the next steps for checking the refund.",
        True,
        "G009"
    ),


    # 4. RETURN REQUEST

    (
        "I want to return the shoes I purchased.",
        "return_request",
        "I can help you understand the return process. Could you provide your order ID so I can guide you through the next steps?",
        True,
        "G010"
    ),

    (
        "These shoes are too small. I want to send them back.",
        "return_request",
        "I'm sorry the shoes don't fit. Could you provide your order ID so I can guide you through the return process?",
        True,
        "G011"
    ),

    (
        "I want to return my order because the shoes arrived damaged.",
        "return_request",
        "I'm sorry your shoes arrived damaged. Could you provide your order ID and briefly describe the damage so I can guide you through the return process?",
        True,
        "G012"
    ),


    # 5. PRODUCT INFORMATION

    (
        "Does this laptop support 32GB of RAM?",
        "product_information",
        "I'd be happy to help you check its memory compatibility. Could you provide the laptop's exact model number?",
        True,
        "G013"
    ),

    (
        "What colors are available for this product?",
        "product_information",
        "I'd be happy to help you check the available colors. Could you share the product name or model?",
        True,
        "G014"
    ),

    (
        "Is this phone compatible with wireless charging?",
        "product_information",
        "Could you share the exact phone model so I can help you check its wireless charging compatibility?",
        True,
        "G015"
    ),


    # 6. TECHNICAL ISSUE

    (
        "Your mobile app crashes every time I open it.",
        "technical_issue",
        "I'm sorry you're experiencing this issue. Which device and operating system are you using?",
        True,
        "G016"
    ),

    (
        "I cannot log into my account even though my password is correct.",
        "technical_issue",
        "I'm sorry you're having trouble logging in. Are you seeing any error message when you try to sign in?",
        True,
        "G017"
    ),

    (
        "The checkout page keeps showing an error when I try to pay.",
        "technical_issue",
        "I'm sorry the checkout page isn't working. Could you share the error message you're seeing?",
        True,
        "G018"
    ),


    # 7. BILLING ISSUE

    (
        "I was charged twice for the same order.",
        "billing_issue",
        "I'm sorry about the duplicate charge. Could you provide your order ID so I can guide you through the appropriate billing support steps?",
        True,
        "G019"
    ),

    (
        "I don't recognize this charge on my account.",
        "billing_issue",
        "I understand your concern. Could you provide the transaction date and amount? Please don't share your full card number or security code.",
        True,
        "G020"
    ),

    (
        "I need a copy of my payment invoice.",
        "billing_issue",
        "I'd be happy to help you find your payment invoice. Could you provide your order ID so I can guide you through the process?",
        True,
        "G021"
    ),


    # 8. COMPLAINT

    (
        "Your customer service is terrible. Nobody has helped me.",
        "complaint",
        "I'm sorry about your experience. Could you tell me what issue you're trying to resolve so I can better understand your situation?",
        True,
        "G022"
    ),

    (
        "I'm extremely disappointed with the quality of your service.",
        "complaint",
        "I'm sorry we haven't met your expectations. Could you tell me more about what went wrong?",
        True,
        "G023"
    ),

    (
        "This is the third time I've contacted support and nobody has resolved my problem.",
        "complaint",
        "I apologize for the repeated inconvenience. Could you briefly describe the unresolved issue and share any existing support reference number? I can help you understand the next steps.",
        True,
        "G024"
    ),


    # 9. OTHER

    (
        "Hello!",
        "other",
        "Hello! How can I help you today?",
        False,
        "G025"
    ),

    (
        "Thank you for your help.",
        "other",
        "You're welcome! Please let me know if you need anything else.",
        False,
        "G026"
    ),

    (
        "Can you explain what your company does?",
        "other",
        "I'd be happy to explain our company. Are you interested in our products, services, or how our business operates?",
        True,
        "G027"
    )

]

Convert the seed tuples into the standard SupportMind message structure using `make_sample`, then report the number of generated records.


In [ ]:
seed_samples = [
    make_sample(
        user_message=user,
        intent=intent,
        response=response,
        needs_clarification=clarification,
        group_id=group_id
    )
    for user, intent, response, clarification, group_id in seed_data
]

print(f"Total samples: {len(seed_samples)}")

Count the examples belonging to each supported intent. This gives a quick view of class balance before splitting and training.


In [ ]:
from collections import Counter

intent_distribution = Counter(
    json.loads(sample["messages"][2]["content"])["intent"]
    for sample in seed_samples
)

for intent, count in sorted(intent_distribution.items()):
  print(f"{intent:22} : {count}")

Validate all 45 combined seed and augmentation examples before splitting. This checks the conversation schema and ensures that every user message and target response contains usable text.


In [ ]:
def validate_dataset(dataset):
  assert len(dataset) > 0

  for index, sample in enumerate(dataset):

    try:
      validate_sample(sample)

      user_message = sample["messages"][1]["content"]
      output = json.loads(sample["messages"][2]["content"])

      assert user_message.strip()
      assert output["response"].strip()

    except Exception as error:
      print(f"Invalid sample at index {index}: {error}")
      raise
  print(f"Dataset validation passed: {len(dataset)} samples")

validate_dataset(seed_samples)

Define additional customer-support examples and paraphrases. Related paraphrases retain the same `group_id`, allowing group-aware splitting to prevent leakage.


In [ ]:
augmentation_data = [

    # ORDER STATUS

    (
        "I'd like to know the current status of my purchase.",
        "order_status",
        "I'd be happy to help you check your order. Could you provide your order ID so I can guide you through the tracking process?",
        True,
        "G002"
    ),

    (
        "Can you track my order number 78921?",
        "order_status",
        "I can help you with your tracking inquiry. I don't have direct access to live tracking information, but you can use your order number on our order tracking page. If you have a separate tracking number, you can use that as well.",
        False,
        "G028"
    ),


    # SHIPPING ISSUE

    (
        "My delivery hasn't arrived on the expected date.",
        "shipping_issue",
        "I'm sorry your delivery is delayed. Could you provide your order ID so I can guide you through the available shipping support options?",
        True,
        "G004"
    ),

    (
        "The courier hasn't delivered my package yet.",
        "shipping_issue",
        "I'm sorry your package hasn't arrived. Could you share your order ID so I can help you understand the next steps for checking its delivery status?",
        True,
        "G029"
    ),


    # REFUND REQUEST

    (
        "How can I get my money back?",
        "refund_request",
        "I can help you understand the refund process. Could you provide your order ID so I can guide you through the next steps?",
        True,
        "G007"
    ),

    (
        "I'd like to cancel my purchase and receive a refund.",
        "refund_request",
        "I can help you understand the cancellation and refund process. Could you provide your order ID so I can guide you through the available options?",
        True,
        "G030"
    ),


    # RETURN REQUEST

    (
        "The shoes don't fit me. How can I return them?",
        "return_request",
        "I'm sorry the shoes don't fit. Could you provide your order ID so I can guide you through the return process?",
        True,
        "G011"
    ),

    (
        "Can I send this product back?",
        "return_request",
        "I'd be happy to help you understand the return process. Could you share the product name and your order ID?",
        True,
        "G031"
    ),


    # PRODUCT INFORMATION

    (
        "What are the dimensions of this product?",
        "product_information",
        "I can help you find the product dimensions. Could you share the product name or model?",
        True,
        "G032"
    ),

    (
        "Do you have this item in other colors?",
        "product_information",
        "I'd be happy to help you check the available colors. Could you share the product name or model?",
        True,
        "G014"
    ),


    # TECHNICAL ISSUE

    (
        "The website stops responding whenever I click checkout.",
        "technical_issue",
        "I'm sorry you're experiencing this issue. Which browser are you using, and do you see any error message?",
        True,
        "G018"
    ),

    (
        "I keep getting an error when trying to reset my password.",
        "technical_issue",
        "I'm sorry you're having trouble resetting your password. Could you share the error message you're seeing?",
        True,
        "G033"
    ),


    # BILLING ISSUE

    (
        "Why was I charged more than the displayed price?",
        "billing_issue",
        "I understand your concern. Could you provide your order ID and the amount you were charged? Please don't share your full card number or security code.",
        True,
        "G034"
    ),

    (
        "My payment was deducted, but the order still shows as unpaid.",
        "billing_issue",
        "I'm sorry about this payment issue. Could you provide your order ID and payment transaction reference? Please don't share your full card number or security code.",
        True,
        "G035"
    ),


    # COMPLAINT

    (
        "I'm really unhappy with the way your team handled my issue.",
        "complaint",
        "I'm sorry about your experience. Could you tell me what happened and which issue you're still having trouble with?",
        True,
        "G022"
    ),

    (
        "This is unacceptable. I've had a terrible experience with your service.",
        "complaint",
        "I'm sorry we haven't met your expectations. Could you tell me what went wrong so I can better understand your concerns?",
        True,
        "G023"
    ),


    # OTHER

    (
        "Do you offer any loyalty programs?",
        "other",
        "I'd be happy to help you find information about our loyalty programs. Are you interested in program eligibility, earning rewards, or redeeming points?",
        True,
        "G036"
    ),

    (
        "Where can I find your company information?",
        "other",
        "I can help you find company information. Are you looking for our company background, contact details, or information about our products and services?",
        True,
        "G037"
    )

]

Convert augmentation tuples into the standard conversational message format, preserving their group identifiers for leakage-safe splitting.


In [ ]:
augmented_samples = [
    make_sample(
        user_message=user,
        intent=intent,
        response=response,
        needs_clarification=clarification,
        group_id=group_id
    )
    for user, intent, response, clarification, group_id in augmentation_data
]

print("Augmented samples:", len(augmented_samples))


Combine the seed and augmented records into one dataset and verify that every record has a group identifier.


In [ ]:
print("Original samples:", len(seed_samples))
print("Augmented samples:", len(augmented_samples))

dataset = seed_samples + augmented_samples

assert all("group_id" in sample for sample in dataset)
assert len(dataset) == len(seed_data) + len(augmentation_data)

print("All samples contain group_id.")
print("Total samples:", len(dataset))


Count how many examples belong to each group. Groups with multiple rows usually represent a seed sample and one or more paraphrases.


In [ ]:
from collections import Counter

group_distribution = Counter(
    sample["group_id"] for sample in dataset
)

print("Total samples:", len(dataset))
print("Unique groups:", len(group_distribution))

for group_id, count in sorted(group_distribution.items()):
    if count > 1:
        print(f"{group_id}: {count} samples")


Count the examples belonging to each supported intent. This gives a quick view of class balance before splitting and training.


In [ ]:
from collections import Counter
import json

intent_distribution = Counter(
    json.loads(sample["messages"][2]["content"])["intent"]
    for sample in dataset
)

for intent in INTENTS:
  print(f"{intent:22}: {intent_distribution[intent]}")

Detect duplicate user messages after normalizing case and whitespace. Duplicate checks help reduce repeated examples and expose accidental copy/paste errors.


In [ ]:
def find_duplicates(dataset):
    seen = {}
    duplicates = []

    for index, sample in enumerate(dataset):
        user_message = " ".join(
            sample["messages"][1]["content"].lower().split()
        )

        if user_message in seen:
            duplicates.append((seen[user_message], index))
        else:
            seen[user_message] = index

    return duplicates


duplicates = find_duplicates(dataset)

print("Duplicate pairs:", duplicates)
print("Duplicate count:", len(duplicates))


Run the 3. dataset engineering step shown in the code. This cell prepares or validates the corresponding artifact for the next stage of the SupportMind fine-tuning workflow.


In [ ]:
validate_dataset(dataset)

Count how many examples require clarification. The current synthetic dataset is intentionally small and has a strong True/False imbalance, so clarification metrics must be interpreted cautiously and the dataset should be expanded before any real-world claim.


In [ ]:
clarification_distribution = Counter(
    json.loads(sample["messages"][2]["content"])["needs_clarification"]
    for sample in dataset
)

print("Needs clarification:", clarification_distribution[True])
print("No clarification:", clarification_distribution[False])

Measure the word-count distribution of customer messages. This is a lightweight dataset inspection; token lengths are measured separately after applying the model tokenizer.


In [ ]:
message_lengths = [
    len(sample["messages"][1]["content"].split())
    for sample in dataset
]

print("Minimum:", min(message_lengths))
print("Maximum:", max(message_lengths))
print("Average:", sum(message_lengths) / len(message_lengths))

Inspect unusually short or long customer messages. This helps identify examples that may be too vague, overly verbose, or inconsistent with the intended task.


In [ ]:
for index, sample in enumerate(dataset):
  message = sample["messages"][1]["content"]
  word_count = len(message.split())

  if word_count < 3 or word_count > 40:
    print(f"Index: {index}")
    print(f"Words: {word_count}")
    print(f"Message: {message}")
    print("-" * 50)

Detect duplicate user messages after normalizing case and whitespace. Duplicate checks help reduce repeated examples and expose accidental copy/paste errors.


In [ ]:
duplicates = find_duplicates(dataset)

print("Duplicate indices:", duplicates)
print("Duplicate count:", len(duplicates))

Install the Python packages required by the following workflow. Package installation is kept in notebook cells so the environment can be reproduced in a fresh Colab runtime.


In [ ]:
import sklearn
print('scikit-learn version:', sklearn.__version__)


Split examples by `group_id` rather than by individual row. This prevents a seed example and its paraphrases from appearing in different splits and causing data leakage.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = [sample["group_id"] for sample in dataset]

# First split: reserve approximately 20% of groups for the test set.
test_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_val_indices, test_indices = next(
    test_splitter.split(dataset, groups=groups)
)

train_val_data = [dataset[index] for index in train_val_indices]
test_data = [dataset[index] for index in test_indices]

print("Train + validation samples:", len(train_val_data))
print("Test samples:", len(test_data))


Split examples by `group_id` rather than by individual row. This prevents a seed example and its paraphrases from appearing in different splits and causing data leakage.


In [ ]:
# Second split: reserve a portion of the remaining groups for validation.
train_val_groups = [sample["group_id"] for sample in train_val_data]

validation_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.125,
    random_state=42
)

train_indices, val_indices = next(
    validation_splitter.split(train_val_data, groups=train_val_groups)
)

train_data = [train_val_data[index] for index in train_indices]
val_data = [train_val_data[index] for index in val_indices]

print("Train samples:", len(train_data))
print("Validation samples:", len(val_data))
print("Test samples:", len(test_data))


Re-check group separation immediately before SFT preparation. These assertions ensure that related paraphrases do not cross the train, validation, and test boundaries.


In [ ]:
train_groups = {sample["group_id"] for sample in train_data}
val_groups = {sample["group_id"] for sample in val_data}
test_groups = {sample["group_id"] for sample in test_data}

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print("No group leakage detected.")
print("Unique groups:", {
    "train": len(train_groups),
    "validation": len(val_groups),
    "test": len(test_groups)
})


### 4. Tokenization & Chat Templates

This section checks the model's chat template, formats conversational messages, tokenizes the complete conversations, and inspects sequence lengths. These checks help confirm that the selected context length is sufficient and that the training examples are compatible with the tokenizer.


Confirm that the tokenizer provides a native chat template. Qwen3's template defines the control tokens and message boundaries used during training and inference.


In [ ]:
# Check tokenizer chat template

assert tokenizer.chat_template is not None

print("Chat template is available.")

Apply the model's native chat template to each conversation so the formatted text matches the structure expected by Qwen3.


In [ ]:
def format_sample(sample):

  text = tokenizer.apply_chat_template(
      sample["messages"],
      tokenize=False,
      add_generation_prompt=False,
      enable_thinking=False
  )


  return {
      "group_id": sample["group_id"],
      "text": text
  }

Apply the Qwen3 chat template separately to the train, validation, and test splits while retaining group IDs only for audit purposes.


In [ ]:
train_formatted = [
    format_sample(sample)
    for sample in train_data
]

val_formatted = [
    format_sample(sample)
    for sample in val_data
]

test_formatted = [
    format_sample(sample)
    for sample in test_data
]

Display the formatted training conversations, including their group IDs, to inspect how the tokenizer's chat template renders each message.


In [ ]:
print(train_formatted)

Display one complete formatted conversation. Inspect the system, user, and assistant boundaries before tokenization.


In [ ]:
print(train_formatted[0]["text"])

Tokenize the formatted conversations without truncation so sequence lengths can be measured before selecting the training context limit.


In [ ]:
def tokenize_samples(samples):

  tokenized_samples = []

  for sample in samples:
      encoded = tokenizer(
        sample["text"],
        add_special_tokens=False,
        truncation=False
      )

      tokenized_samples.append({
          "group_id": sample["group_id"],
          "input_ids": encoded["input_ids"],
          "attention_mask": encoded["attention_mask"]
      })

  return tokenized_samples

Tokenize all formatted splits without truncation. The resulting token IDs and attention masks are used for length analysis and validation.


In [ ]:
train_tokenized = tokenize_samples(train_formatted)

val_tokenized = tokenize_samples(val_formatted)

test_tokenized = tokenize_samples(test_formatted)

Inspect one tokenized example, including its token IDs, attention mask, and group ID, to verify the tokenizer output structure.


In [ ]:
print(train_tokenized[0])

Measure minimum, maximum, mean, and median token lengths for each split. This confirms whether the configured maximum sequence length is adequate.


In [ ]:
import numpy as np

def analyze_token_lengths(samples, name):

  lengths = [
      len(sample["input_ids"])
      for sample in samples
  ]

  print(f"\n{name}")
  print("-" * 30)
  print("Samples:", len(lengths))
  print("Min:", min(lengths))
  print("Max:", max(lengths))
  print("Mean:", round(np.mean(lengths), 2))
  print("Median:", np.median(lengths))

analyze_token_lengths(train_tokenized, "Train")
analyze_token_lengths(val_tokenized, "Validation")
analyze_token_lengths(test_tokenized, "Test")

Check tokenized records for non-empty input IDs, matching attention-mask lengths, and valid binary mask values before training.


In [ ]:
def validate_tokenized_dataset(samples, name):

    errors = []

    for index, sample in enumerate(samples):

        if len(sample["input_ids"]) != len(sample["attention_mask"]):
            errors.append(f"Sample {index}: Length mismatch")

        if len(sample["input_ids"]) == 0:
            errors.append(f"Sample {index}: Empty input_ids")

        if not all(
            mask in (0, 1)
            for mask in sample["attention_mask"]
        ):
            errors.append(f"Sample {index}: Invalid attention_mask")

    print(f"\n{name}")
    print("Total samples:", len(samples))
    print("Errors:", len(errors))

    for error in errors:
        print(error)

    return len(errors) == 0

Check tokenized records for non-empty input IDs, matching attention-mask lengths, and valid binary mask values before training.


In [ ]:
train_valid = validate_tokenized_dataset(
    train_tokenized, "Train"
)

val_valid = validate_tokenized_dataset(
    val_tokenized, "Validation"
)

test_valid = validate_tokenized_dataset(
    test_tokenized, "Test"
)

assert train_valid and val_valid and test_valid

This cell probes the tokenizer's original assistant-token mask support. TRL's SFTTrainer can automatically use a training-compatible patched chat template for recognized Qwen3 families, adding generation markers required by `assistant_only_loss=True`. The probe is diagnostic; the trainer performs its own template handling during initialization.


In [ ]:
sample = train_data[0]

probe = tokenizer.apply_chat_template(
    sample["messages"],
    tokenize=True,
    add_generation_prompt=False,
    return_dict=True,
    return_assistant_tokens_mask=True,
    enable_thinking=False
)

print("Available keys:", list(probe.keys()))

assistant_mask = probe.get("assistant_masks")

if assistant_mask is None:
    print("Assistant token mask is unavailable for this tokenizer/template.")
else:
    print("Input token count:", len(probe["input_ids"]))
    print("Assistant mask length:", len(assistant_mask))
    print("Assistant tokens:", sum(assistant_mask))

    assert len(probe["input_ids"]) == len(assistant_mask)
    assert sum(assistant_mask) > 0, "No assistant tokens were marked for loss."


### 5. Model Loading & Precision Analysis

This section inspects Qwen3-4B's configuration and estimates the memory required by different numerical precisions. The estimates cover model weights only; runtime memory also includes activations, temporary buffers, optimizer states, and other allocations. Actual weight loading is optional.


Retrieve the model configuration from Hugging Face without loading the model weights. The configuration exposes architecture dimensions and context-related metadata used in later memory and model-loading decisions.


In [ ]:
from transformers import AutoConfig

MODEL_NAME = "Qwen/Qwen3-4B"

# This downloads configuration metadata only, not the model weight files.
model_config = AutoConfig.from_pretrained(MODEL_NAME)

print("Architecture:", model_config.architectures)
print("Model type:", model_config.model_type)
print("Hidden size:", model_config.hidden_size)
print("Layers:", model_config.num_hidden_layers)
print("Attention heads:", model_config.num_attention_heads)
print("KV heads:", model_config.num_key_value_heads)
print("Vocabulary size:", model_config.vocab_size)
print("Maximum positions:", model_config.max_position_embeddings)


Inspect the available PyTorch/CUDA runtime and GPU capabilities. The results help verify that the selected FP16 training path is compatible with the current accelerator.


In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)

    print("GPU:", gpu.name)
    print(f"Total VRAM: {gpu.total_memory / 1024**3:.2f} GiB")
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    print("FP16 supported: True")
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    print("CUDA is unavailable.")


Estimate weight-only memory for FP32, FP16, INT8, and 4-bit storage. These calculations are theoretical lower bounds and do not include runtime overhead.


In [ ]:
# Approximate weight-only memory for a 4-billion-parameter model.
# This excludes quantization metadata, activations, optimizer states, and KV cache.

PARAM_COUNT = 4_000_000_000

PRECISIONS = {
    "FP32": 4,
    "FP16": 2,
    "INT8": 1,
    "4-bit": 0.5
}

for precision, bytes_per_parameter in PRECISIONS.items():
    memory_gib = PARAM_COUNT * bytes_per_parameter / (1024 ** 3)
    print(f"{precision}: {memory_gib:.2f} GiB")


Optionally load the base model in FP16 and measure its memory footprint. This experiment illustrates the cost of full-precision weights and releases the model afterward to avoid retaining GPU allocations.


In [ ]:
# Optional FP16 loading experiment. This downloads the full base-model weights.
# It is disabled by default because the notebook uses QLoRA for training.

RUN_FP16_EXPERIMENT = False

if RUN_FP16_EXPERIMENT:
    from unsloth import FastLanguageModel

    fp16_model, fp16_tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=512,
        dtype=torch.float16,
        load_in_4bit=False,
        load_in_8bit=False,
        load_in_16bit=True
    )

    print("FP16 model loaded.")
    print(f"Memory footprint: {fp16_model.get_memory_footprint() / 1024**3:.2f} GiB")

    del fp16_model, fp16_tokenizer
    torch.cuda.empty_cache()
else:
    print("FP16 model loading skipped.")


Optionally load the base model with 8-bit quantization and inspect its memory footprint. The experiment is disabled by default and cleans up the loaded model after measurement.


In [ ]:
# Optional 8-bit loading experiment. Enable it only when you want to compare
# the memory footprint and runtime behavior with the 4-bit QLoRA configuration.

RUN_INT8_EXPERIMENT = False

if RUN_INT8_EXPERIMENT:
    from unsloth import FastLanguageModel

    int8_model, int8_tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=512,
        dtype=torch.float16,
        load_in_4bit=False,
        load_in_8bit=True,
        load_in_16bit=False
    )

    print("INT8 model loaded.")
    print(f"Memory footprint: {int8_model.get_memory_footprint() / 1024**3:.2f} GiB")

    del int8_model, int8_tokenizer
    torch.cuda.empty_cache()
else:
    print("INT8 model loading skipped.")


Record the chosen quantization mode before loading the model. The actual model load occurs in the next cell and is gated by the model-download flag.


In [ ]:
# The training configuration for this lab is 4-bit NF4 QLoRA.
# The actual Unsloth model load is performed after the quantization configuration section.
print("Selected training mode: 4-bit NF4 QLoRA")


##### Actual Model Memory Footprint


Use Unsloth's model-loading API with the selected precision or quantization settings. This cell is an optional experiment unless it is the canonical 4-bit model-loading step.


In [ ]:
import torch
import inspect
import unsloth

from unsloth import FastLanguageModel

print("Unsloth version:", unsloth.__version__)
print("PyTorch version:", torch.__version__)

print("\nFastLanguageModel:")
print(FastLanguageModel)

print("\nfrom_pretrained signature:")
print(inspect.signature(FastLanguageModel.from_pretrained))

### 6. Quantization Experiments (FP16 → INT8 → 4-bit)

Quantization stores model weights with fewer bits to reduce memory consumption. This section compares theoretical weight memory and defines the NF4 configuration used for QLoRA. The actual training model is loaded in 4-bit mode only once, avoiding repeated downloads and unnecessary GPU allocations.


Define and inspect the BitsAndBytes NF4 configuration, including FP16 compute, double quantization, and packed 4-bit storage. The actual Unsloth load uses its corresponding `load_in_4bit` option.


In [ ]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.uint8
)

print(bnb_config)


Estimate weight-only memory for FP32, FP16, INT8, and 4-bit storage. These calculations are theoretical lower bounds and do not include runtime overhead.


In [ ]:
# Theoretical weight-only comparison. Actual VRAM use will be higher.
PARAM_COUNT = 4_000_000_000

precision_bits = {
    "FP32": 32,
    "FP16": 16,
    "INT8": 8,
    "NF4": 4
}

for name, bits in precision_bits.items():
    memory_gib = PARAM_COUNT * bits / 8 / (1024 ** 3)
    print(f"{name}: {memory_gib:.2f} GiB")


Summarize the selected NF4 quantization settings without loading model weights. These settings document the intended QLoRA configuration.


In [ ]:
# Quantization configuration summary; no model weights are loaded in this cell.
quantization_settings = {
    "quantization_type": "NF4",
    "compute_dtype": "float16",
    "double_quantization": True,
    "quant_storage": "uint8"
}

for key, value in quantization_settings.items():
    print(f"{key}: {value}")


Load the Qwen3-4B base model through Unsloth in 4-bit mode when model execution is explicitly enabled. The default setting skips weight downloads, so the remaining data and configuration sections can be studied without consuming GPU memory.


In [ ]:
# Load the 4-bit base model only when you are ready to download its weights.
# This is the model instance that will receive the LoRA adapters.

if RUN_MODEL_DOWNLOADS:
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required for QLoRA.")

    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=512,
        dtype=torch.float16,
        load_in_4bit=True
    )

    MODEL_READY = True
    print("4-bit QLoRA base model loaded.")
else:
    print("Model weights were not downloaded. Set RUN_MODEL_DOWNLOADS=True to load Qwen3-4B.")


Inspect the actual memory footprint of the loaded 4-bit model. This complements the theoretical weight-only estimates from the precision analysis.


In [ ]:
if MODEL_READY:
    memory_bytes = model.get_memory_footprint()
    print("Model memory:", round(memory_bytes / (1024 ** 3), 2), "GiB")
else:
    print("Load the 4-bit model to inspect its actual memory footprint.")


### 7. LoRA Configuration

LoRA (Low-Rank Adaptation) freezes the pretrained weight matrix and learns two smaller matrices, A and B, to represent a low-rank update. This keeps the number of trainable parameters much smaller than full fine-tuning.

For a weight matrix W with shape `(out_features, in_features)`, the adapter matrices have shapes A = `(rank, in_features)` and B = `(out_features, rank)`. The forward computation is:

`y = W @ x + (alpha / rank) * B @ (A @ x)`

The next cell implements this equation with NumPy for learning purposes. The actual model adaptation is performed later with Unsloth's `FastLanguageModel.get_peft_model`.


Implement the LoRA forward equation with NumPy to make the low-rank matrix shapes and zero-initialized adapter update concrete. This is an educational demonstration; Unsloth performs the real model adaptation later.


In [ ]:
import numpy as np

class LoRA:
    def __init__(self, weight: np.ndarray, rank: int, alpha: float):
        if weight.ndim != 2:
            raise ValueError("weight must be a 2D matrix.")

        self.weight = weight
        self.out_features, self.in_features = weight.shape
        self.rank = rank
        self.alpha = alpha

        # A is randomly initialized; B starts at zero.
        self.A = np.random.normal(
            loc=0.0,
            scale=0.01,
            size=(rank, self.in_features)
        )
        self.B = np.zeros((self.out_features, rank))

    def forward(self, x: np.ndarray) -> np.ndarray:
        base_output = self.weight @ x
        adapter_output = self.B @ (self.A @ x)
        scaling = self.alpha / self.rank
        return base_output + scaling * adapter_output


# A small shape test: W(out, in), A(rank, in), B(out, rank), x(in,).
demo_weight = np.random.randn(6, 4)
demo_input = np.random.randn(4)

lora_demo = LoRA(weight=demo_weight, rank=2, alpha=4)
demo_output = lora_demo.forward(demo_input)

print("Weight shape:", lora_demo.weight.shape)
print("A shape:", lora_demo.A.shape)
print("B shape:", lora_demo.B.shape)
print("Input shape:", demo_input.shape)
print("Output shape:", demo_output.shape)

# Since B is initialized to zero, the initial adapter update is exactly zero.
assert np.allclose(demo_output, demo_weight @ demo_input)
print("LoRA initialization check passed.")


Centralize the LoRA hyperparameters so the adapter configuration is visible, reproducible, and reusable by Unsloth's PEFT integration.


In [ ]:
LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    "bias": "none",
    "use_gradient_checkpointing": "unsloth",
    "random_state": 3407,
    "use_rslora": False
}

for key, value in LORA_CONFIG.items():
    print(f"{key}: {value}")


Attach LoRA adapters to the loaded 4-bit base model using the centralized configuration. This is the actual Unsloth/PEFT adaptation step; it is skipped when model weights are not loaded.


In [ ]:
# Attach trainable LoRA adapters to the frozen 4-bit base model.
# The model must have been loaded in the previous cell.

if MODEL_READY:
    model = FastLanguageModel.get_peft_model(
        model,
        **LORA_CONFIG
    )
    print("LoRA adapters attached.")
else:
    print("LoRA attachment skipped because model weights are not loaded.")


##### Trainable Parameters

Report how many parameters are trainable versus total model parameters. This verifies that the run is adapter fine-tuning rather than full-model fine-tuning.


In [ ]:
if MODEL_READY:
    model.print_trainable_parameters()
else:
    print("Trainable parameter counts are available after loading the model and attaching LoRA.")


### 8. SFT Dataset Preparation

This section converts the leakage-safe Python splits into Hugging Face `Dataset` objects in conversational `messages` format. It also checks the installed TRL API, validates the dataset schema, and defines the SFT training configuration. The auxiliary `group_id` is deliberately excluded from model inputs.


Install the Python packages required by the following workflow. Package installation is kept in notebook cells so the environment can be reproduced in a fresh Colab runtime.


In [ ]:
%pip install -q datasets


Convert Python conversation records into Hugging Face Dataset objects. Only the `messages` field is retained; group identifiers remain outside the model's training input.


In [ ]:
from datasets import Dataset

def prepare_sft_dataset(samples):
    # Keep only the conversational messages. group_id is for split auditing,
    # not a token the model should learn to generate.
    records = [
        {"messages": sample["messages"]}
        for sample in samples
    ]
    return Dataset.from_list(records)


Convert Python conversation records into Hugging Face Dataset objects. Only the `messages` field is retained; group identifiers remain outside the model's training input.


In [ ]:
train_sft_dataset = prepare_sft_dataset(train_data)
val_sft_dataset = prepare_sft_dataset(val_data)
test_sft_dataset = prepare_sft_dataset(test_data)

print("Hugging Face conversational datasets created.")


Inspect the size, feature schema, and first record of the Hugging Face datasets to confirm that each split remains in conversational `messages` format.


In [ ]:
print("Train:", len(train_sft_dataset))
print("Validation:", len(val_sft_dataset))
print("Test:", len(test_sft_dataset))

print("\nDataset Features:")
print(train_sft_dataset.features)

print("\nFirst Sample:")
print(train_sft_dataset[0])

Install the Python packages required by the following workflow. Package installation is kept in notebook cells so the environment can be reproduced in a fresh Colab runtime.


In [ ]:
import trl
print("TRL version:", trl.__version__)
print("Transformers version:", transformers.__version__)


Inspect the installed TRL API signatures instead of assuming parameter names. This is useful when library versions change and prevents unsupported configuration arguments.


In [ ]:
import inspect

from trl import SFTTrainer, SFTConfig

print("SFTTrainer Signature:")
print(inspect.signature(SFTTrainer.__init__))

print("\nSFTConfig Signature:")
print(inspect.signature(SFTConfig))

Record the installed TRL and Transformers versions. These versions determine which SFTTrainer and SFTConfig arguments are available.


In [ ]:
import trl

print("TRL Version:", trl.__version__)
print("Transformers Version:", __import__("transformers").__version__)

Re-check group separation immediately before SFT preparation. These assertions ensure that related paraphrases do not cross the train, validation, and test boundaries.


In [ ]:
train_groups = {sample["group_id"] for sample in train_data}
val_groups = {sample["group_id"] for sample in val_data}
test_groups = {sample["group_id"] for sample in test_data}

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print("Train ∩ Validation:", train_groups & val_groups)
print("Train ∩ Test:", train_groups & test_groups)
print("Validation ∩ Test:", val_groups & test_groups)


Check that every configured SFT argument exists in the installed TRL version, then construct the SFTConfig. This catches API-version mismatches before trainer initialization.


In [ ]:
from trl import SFTConfig
import inspect

# These arguments were checked against the installed TRL version in this lab.
supported_sft_args = set(inspect.signature(SFTConfig.__init__).parameters)

SFT_CONFIG = {
    "output_dir": "./outputs/supportmind",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 1e-4,
    "weight_decay": 0.01,
    "warmup_steps": 1,
    "lr_scheduler_type": "cosine",
    "logging_steps": 1,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "fp16": True,
    "bf16": False,
    "gradient_checkpointing": True,
    "max_length": 512,
    "packing": False,
    "assistant_only_loss": True,
    "report_to": "none",
    "seed": 3407
}

unsupported_args = set(SFT_CONFIG) - supported_sft_args
if unsupported_args:
    raise TypeError(
        f"The installed TRL version does not support these SFTConfig arguments: {unsupported_args}"
    )

sft_config = SFTConfig(**SFT_CONFIG)

print("SFTConfig created successfully.")


Display the fully instantiated SFTConfig so the effective training settings can be reviewed before creating the trainer.


In [ ]:
print(sft_config)

##### Data Quality Audit

Validate each sample's message roles, JSON structure, supported intent, response type, and clarification flag. Early schema checks prevent malformed examples from silently entering training.


In [ ]:
import json

def validate_sample_schema(sample):
    messages = sample.get("messages", [])

    if len(messages) != 3:
        return False, "Expected exactly three messages."

    if [message.get("role") for message in messages] != [
        "system", "user", "assistant"
    ]:
        return False, "Unexpected message role order."

    try:
        response = json.loads(messages[-1]["content"])
    except json.JSONDecodeError:
        return False, "Invalid assistant JSON."

    if not isinstance(response, dict):
        return False, "Assistant output must be a JSON object."

    required_fields = {
        "intent",
        "response",
        "needs_clarification"
    }

    if set(response) != required_fields:
        return False, "Invalid output fields."

    if response["intent"] not in INTENTS:
        return False, "Invalid intent."

    if not isinstance(response["response"], str) or not response["response"].strip():
        return False, "Invalid response value."

    if not isinstance(response["needs_clarification"], bool):
        return False, "Invalid clarification value."

    if not messages[1]["content"].strip():
        return False, "Empty user message."

    return True, "Valid"


for split_name, split_data in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data)
]:
    results = [validate_sample_schema(sample) for sample in split_data]
    valid_count = sum(is_valid for is_valid, _ in results)

    print(f"\n{split_name}: {valid_count}/{len(split_data)} valid")

    for index, (is_valid, reason) in enumerate(results):
        if not is_valid:
            print(f"Sample {index}: {reason}")

    assert valid_count == len(split_data), f"{split_name} contains invalid samples."


##### Semantic Quality Audit

Print all examples in each split for a human semantic audit. Review whether the intent matches the customer's primary requested action and whether the response asks only for information that is actually missing.


In [ ]:
for split_name, dataset in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data)
]:
    print(f"\n{'=' * 20} {split_name} {'=' * 20}")

    for index, sample in enumerate(dataset):
      user_message = sample["messages"][1]["content"]
      assistant_response = json.loads(
          sample["messages"][2]["content"]
      )

      print(f"\nSample {index + 1}")
      print("User:", user_message)
      print("Intent:", assistant_response["intent"])
      print("Response:", assistant_response["response"])
      print(
          "Needs Clarification:",
          assistant_response["needs_clarification"]
      )

**Dataset limitation:** this lab uses 45 synthetic examples and only 11 test examples. It is suitable for learning the fine-tuning workflow and debugging the pipeline, but it is not large or representative enough to support claims about production customer-support performance. A real deployment requires consented, reviewed, diverse customer-support data and a substantially larger evaluation set.


### 9. Supervised Fine-Tuning

Base Model (Qwen3-4B)  --->  Quantized Model (4-bit NF4)  --->  LoRA Adapters  --->  SFTTrainer  --->  trainer.train()  --->  Fine-tuned SupportMind

Initialize TRL's SFTTrainer with the Unsloth model, SFTConfig, conversational datasets, and tokenizer. For recognized Qwen3 families, TRL can apply a training-compatible chat template containing generation markers. The code verifies those markers before allowing assistant-only loss training.


In [ ]:
from trl import SFTTrainer

trainer = None

if MODEL_READY:
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_sft_dataset,
        eval_dataset=val_sft_dataset,
        processing_class=tokenizer
    )

    # TRL patches recognized Qwen3 templates for assistant-only loss.
    training_tokenizer = getattr(trainer, "processing_class", tokenizer)
    training_template = getattr(training_tokenizer, "chat_template", None) or ""

    if trainer.args.assistant_only_loss:
        has_generation_markers = (
            "{% generation %}" in training_template
            and "{% endgeneration %}" in training_template
        )
        if not has_generation_markers:
            raise RuntimeError(
                "assistant_only_loss=True requires generation markers in the "
                "training chat template. Inspect the installed TRL version "
                "and its Qwen3 template patching behavior."
            )

    print("SFTTrainer initialized.")
else:
    print("SFTTrainer initialization skipped because model weights are not loaded.")


Inspect the trainer's effective dataset sizes, batch settings, gradient accumulation, epochs, and precision flags before starting training.


In [ ]:
if trainer is not None:
    print("Training samples:", len(trainer.train_dataset))
    print("Validation samples:", len(trainer.eval_dataset))
    print("Per-device train batch size:", trainer.args.per_device_train_batch_size)
    print("Gradient accumulation steps:", trainer.args.gradient_accumulation_steps)
    print("Effective batch size:", (
        trainer.args.per_device_train_batch_size
        * trainer.args.gradient_accumulation_steps
    ))
    print("Epochs:", trainer.args.num_train_epochs)
    print("FP16:", trainer.args.fp16)
    print("BF16:", trainer.args.bf16)
else:
    print("Trainer settings will be available after SFTTrainer initialization.")


Run supervised fine-tuning only when both the model and training flags are enabled. The call updates LoRA parameters and records training/evaluation metrics.


In [ ]:
if trainer is not None and RUN_TRAINING:
    training_results = trainer.train()
    TRAINING_COMPLETED = True
    print(training_results)
else:
    print("Training skipped. Set RUN_MODEL_DOWNLOADS=True and RUN_TRAINING=True to run SFT.")


During supervised fine-tuning, each batch is passed through the model and a loss is calculated against the target assistant tokens. Backpropagation computes gradients for trainable LoRA parameters, while the quantized base weights remain frozen. Gradient accumulation combines gradients across multiple micro-batches before the optimizer performs an update. The learning-rate scheduler controls the update scale during training.


### 10. Training Analysis

This section inspects the training log history and plots training loss against validation loss. Training loss indicates how well the model fits the training examples; validation loss helps identify whether that improvement generalizes or whether overfitting may be occurring.


Read the Trainer's recorded metrics, including training loss, evaluation loss, learning rate, epoch, and step information.


In [ ]:
if trainer is not None:
    logs = trainer.state.log_history
    for log in logs:
        print(log)
else:
    logs = []
    print("No training logs are available because training was not run.")


Plot training and validation loss from the Trainer log history. Diverging curves can indicate overfitting, while both curves decreasing suggests the model is still improving on the observed data.


In [ ]:
import matplotlib.pyplot as plt

train_epochs = []
train_losses = []
eval_epochs = []
eval_losses = []

for log in logs:
    if "loss" in log and "eval_loss" not in log:
        train_epochs.append(log.get("epoch"))
        train_losses.append(log["loss"])

    if "eval_loss" in log:
        eval_epochs.append(log.get("epoch"))
        eval_losses.append(log["eval_loss"])

if train_losses or eval_losses:
    plt.figure(figsize=(10, 5))

    if train_losses:
        plt.plot(train_epochs, train_losses, marker="o", label="Training loss")

    if eval_losses:
        plt.plot(eval_epochs, eval_losses, marker="o", label="Validation loss")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("SupportMind Training Analysis")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("No loss values to plot. Run SFT training first.")


Inspect the final Trainer state, including optimizer steps and epoch progress, to confirm how much training actually ran.


In [ ]:
if trainer is not None:
    print("Trainer state type:", type(trainer.state))
    print("Global optimizer steps:", trainer.state.global_step)
    print("Current epoch:", trainer.state.epoch)
else:
    print("Trainer state is unavailable.")


### 11. Model Evaluation


#### Evaluation Metrics

The following metrics evaluate the performance of the customer-support assistant across intent detection, output format, clarification behavior, and final response quality.

| Metric | What It Evaluates |
|---|---|
| **Intent Accuracy** | Whether the model correctly identifies the customer’s intent. |
| **JSON Validity** | Whether the model’s output is valid JSON. |
| **Clarification Accuracy** | Whether the model correctly determines when it needs additional information from the customer. |
| **Response Quality** | Whether the generated response is appropriate, accurate, and helpful. |

#### Overall Accuracy

Overall Accuracy measures the proportion of test cases in which the model produced the expected result.

Accuracy = Correct Predictions / Total Predictions

In this notebook, intent accuracy, clarification accuracy, and JSON validity are reported separately rather than hiding different failure modes inside one combined score. Response quality is reviewed manually because relevance, factuality, clarity, and safety require human judgment.

Switch the loaded Unsloth model into inference mode. This prepares the model for generation and avoids training-oriented behavior during evaluation.


In [ ]:
from unsloth import FastLanguageModel

if MODEL_READY:
    FastLanguageModel.for_inference(model)
    model.eval()
    print("Model switched to inference mode.")
else:
    print("Inference preparation skipped because model weights are not loaded.")


Generate one prediction for a held-out sample. The function sends only system and user messages, applies the Qwen chat template, generates tokens, and removes the original prompt tokens before decoding.


In [ ]:
import torch

def generate_prediction(sample, model, tokenizer, max_new_tokens=256):
    # Only the system and user turns are provided at inference time.
    input_messages = [
        message
        for message in sample["messages"]
        if message["role"] in ("system", "user")
    ]

    prompt = tokenizer.apply_chat_template(
        input_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(model.device)

    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    prompt_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][prompt_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


Generate predictions for the held-out test split and store each raw output alongside its ground-truth annotation. This creates the shared record used by the evaluation metrics.


In [ ]:
predictions = []

if MODEL_READY and TRAINING_COMPLETED and RUN_EVALUATION:
    for sample in test_data:
        raw_output = generate_prediction(
            sample=sample,
            model=model,
            tokenizer=tokenizer
        )

        expected_output = json.loads(sample["messages"][-1]["content"])

        predictions.append({
            "group_id": sample["group_id"],
            "user_message": sample["messages"][1]["content"],
            "expected": expected_output,
            "raw_prediction": raw_output
        })

    print("Total predictions:", len(predictions))
else:
    print(
        "Model evaluation skipped. Complete training and set "
        "RUN_EVALUATION=True to generate test predictions."
    )


Validate generated model output independently from dataset validation. It checks strict JSON parsing, exact fields, supported intents, and the expected Python value types.


In [ ]:
def validate_prediction(raw_output):
    try:
        parsed = json.loads(raw_output)
    except json.JSONDecodeError:
        return False, None

    if not isinstance(parsed, dict):
        return False, None

    required_fields = {
        "intent",
        "response",
        "needs_clarification"
    }

    if set(parsed) != required_fields:
        return False, None

    if not isinstance(parsed["intent"], str) or parsed["intent"] not in INTENTS:
        return False, None

    if not isinstance(parsed["response"], str) or not parsed["response"].strip():
        return False, None

    if not isinstance(parsed["needs_clarification"], bool):
        return False, None

    return True, parsed


Apply the output validator to every prediction and store both validity status and parsed structured output for downstream metrics.


In [ ]:
for item in predictions:
    is_valid, parsed_output = validate_prediction(item["raw_prediction"])
    item["json_valid"] = is_valid
    item["parsed_prediction"] = parsed_output


Import standard classification metrics from scikit-learn for intent and clarification evaluation.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report


Calculate the fraction of test predictions that satisfy the exact JSON output contract. Invalid outputs remain failures rather than being silently repaired.


In [ ]:
total_samples = len(predictions)

valid_json_count = sum(
    item["json_valid"]
    for item in predictions
)

json_validity = (
    valid_json_count / total_samples
    if total_samples
    else 0.0
)

print(f"Valid JSON: {valid_json_count}/{total_samples}")
print(f"JSON validity: {json_validity:.3f}")


Build the ground-truth and predicted intent arrays and calculate intent accuracy. Invalid JSON outputs receive a dedicated invalid label and count as incorrect.


In [ ]:
y_true_intent = [
    item["expected"]["intent"]
    for item in predictions
]

y_pred_intent = [
    item["parsed_prediction"]["intent"]
    if item["json_valid"]
    else "__invalid__"
    for item in predictions
]

intent_accuracy = (
    accuracy_score(y_true_intent, y_pred_intent)
    if predictions
    else 0.0
)

print(f"Intent accuracy: {intent_accuracy:.3f}")


Print per-intent precision, recall, and F1. The invalid-output label is included so malformed predictions remain visible in the report.


In [ ]:
if predictions:
    report_labels = INTENTS + ["__invalid__"]

    print(
        classification_report(
            y_true_intent,
            y_pred_intent,
            labels=report_labels,
            zero_division=0
        )
    )
else:
    print("Classification report unavailable: no predictions were generated.")


Compare expected and predicted clarification decisions. Invalid JSON outputs count as incorrect rather than being excluded from the metric.


In [ ]:
y_true_clarification = [
    item["expected"]["needs_clarification"]
    for item in predictions
]

y_pred_clarification = [
    item["parsed_prediction"]["needs_clarification"]
    if item["json_valid"]
    else None
    for item in predictions
]

clarification_accuracy = (
    sum(
        expected == predicted
        for expected, predicted in zip(
            y_true_clarification,
            y_pred_clarification
        )
    ) / len(predictions)
    if predictions
    else 0.0
)

print(f"Clarification accuracy: {clarification_accuracy:.3f}")


Calculate the fraction of test predictions that satisfy the exact JSON output contract. Invalid outputs remain failures rather than being silently repaired.


In [ ]:
evaluation_results = {
    "total_samples": total_samples,
    "valid_json": valid_json_count,
    "invalid_json": total_samples - valid_json_count,
    "json_validity": json_validity,
    "intent_accuracy": intent_accuracy,
    "clarification_accuracy": clarification_accuracy
}

for metric, value in evaluation_results.items():
    print(f"{metric}: {value}")


Display each test example, its expected structured response, and the model's raw and parsed prediction for manual error analysis.


In [ ]:
for item in predictions:
    print("=" * 60)
    print("Group:", item["group_id"])
    print("User:", item["user_message"])

    print("\nExpected:")
    print(item["expected"])

    print("\nRaw prediction:")
    print(item["raw_prediction"])

    print("\nJSON valid:", item["json_valid"])
    print("Parsed prediction:", item["parsed_prediction"])


#### Manual Response-Quality Review

Automatic metrics cannot reliably judge whether a customer-support response is relevant, safe, clear, and factually grounded. This section creates a review sheet for human evaluation. Rate each dimension from 1 (poor) to 5 (excellent), and record unsupported or uncertain claims in the notes. Do not fill in scores automatically; the ratings must come from reviewing the generated responses.


Create and save a blank human-review worksheet for response relevance, factuality, clarity, and safety. Ratings must be supplied by a reviewer; the notebook does not fabricate subjective quality scores.


In [ ]:
import os
import pandas as pd

quality_review = pd.DataFrame([
    {
        "group_id": item["group_id"],
        "user_message": item["user_message"],
        "predicted_response": (
            item["parsed_prediction"]["response"]
            if item["json_valid"]
            else item["raw_prediction"]
        ),
        "relevance_1_to_5": None,
        "factuality_1_to_5": None,
        "clarity_1_to_5": None,
        "safety_1_to_5": None,
        "review_notes": ""
    }
    for item in predictions
])

print("Human review rows:", len(quality_review))
display(quality_review.head())

QUALITY_REVIEW_PATH = "./outputs/supportmind/response_quality_review.csv"
os.makedirs(os.path.dirname(QUALITY_REVIEW_PATH), exist_ok=True)
quality_review.to_csv(QUALITY_REVIEW_PATH, index=False)
print("Review template saved to:", QUALITY_REVIEW_PATH)


### 12. Adapter Management & Export

This section saves the trained LoRA adapter and tokenizer, records experiment metadata, optionally reloads the adapter as a smoke test, and optionally merges the adapter into a standalone 16-bit model. Adapter-only storage is smaller and preserves the separation between the base model and learned parameters; a merged model is more convenient for deployment but requires additional memory and disk space.


#### Summarize Completed Human Reviews

After a reviewer fills the CSV rating columns and saves the file, run the next cell to calculate average scores. Empty ratings are ignored, and no aggregate is reported until at least one rating has been entered.


In [ ]:
reviewed_quality = pd.read_csv(QUALITY_REVIEW_PATH)

rating_columns = [
    "relevance_1_to_5",
    "factuality_1_to_5",
    "clarity_1_to_5",
    "safety_1_to_5"
]

for column in rating_columns:
    reviewed_quality[column] = pd.to_numeric(
        reviewed_quality[column],
        errors="coerce"
    )

available_scores = reviewed_quality[rating_columns].stack()

if available_scores.empty:
    print("No human ratings have been entered yet.")
else:
    print("Average human-review scores:")
    print(reviewed_quality[rating_columns].mean().round(2))
    print("Overall mean:", round(available_scores.mean(), 2))


Save the trained LoRA adapter and tokenizer to a dedicated directory. This stores the learned adapter separately from the large base-model weights.


In [ ]:
import os

ADAPTER_DIR = "./outputs/supportmind/final_adapter"

if MODEL_READY and TRAINING_COMPLETED and RUN_EXPORT:
    os.makedirs(ADAPTER_DIR, exist_ok=True)

    # Save the trained LoRA adapter and its tokenizer.
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)

    print("Adapter and tokenizer saved to:", ADAPTER_DIR)
else:
    print("Adapter saving skipped. Complete training and set RUN_EXPORT=True.")


Inspect the exported adapter directory and report file sizes. This confirms which artifacts were actually written to disk.


In [ ]:
if os.path.isdir(ADAPTER_DIR):
    for root, _, filenames in os.walk(ADAPTER_DIR):
        print("Directory:", root)

        for filename in filenames:
            file_path = os.path.join(root, filename)
            size_mb = os.path.getsize(file_path) / (1024 ** 2)
            print(f"  {filename}: {size_mb:.2f} MB")
else:
    print("Adapter directory does not exist yet.")


Save reproducibility metadata alongside the adapter, including model identity, quantization, LoRA/SFT settings, dataset split sizes, and software versions.


In [ ]:
import json
import torch
import unsloth
import transformers
import trl
import hashlib

_dataset_payload = json.dumps(
    [sample["messages"] for sample in dataset],
    ensure_ascii=False,
    sort_keys=True
).encode("utf-8")
_dataset_sha256 = hashlib.sha256(_dataset_payload).hexdigest()

experiment_metadata = {
    "project": "SupportMind",
    "base_model": MODEL_NAME,
    "training_method": "QLoRA + SFT",
    "quantization": {
        "type": "NF4",
        "bits": 4,
        "double_quant": True,
        "compute_dtype": "float16"
    },
    "lora_config": LORA_CONFIG,
    "sft_config": SFT_CONFIG,
    "dataset": {
        "total_samples": len(dataset),
        "sha256": _dataset_sha256,
        "train_samples": len(train_data),
        "validation_samples": len(val_data),
        "test_samples": len(test_data),
        "unique_groups": len({sample["group_id"] for sample in dataset})
    },
    "versions": {
        "torch": str(torch.__version__),
        "transformers": transformers.__version__,
        "trl": trl.__version__,
        "unsloth": unsloth.__version__
    }
}

if os.path.isdir(ADAPTER_DIR):
    metadata_path = os.path.join(ADAPTER_DIR, "experiment_metadata.json")

    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(
            experiment_metadata,
            file,
            indent=4,
            ensure_ascii=False,
            default=str
        )

    print("Experiment metadata saved:", metadata_path)
else:
    print("Metadata will be saved after the adapter directory is created.")


Optionally reload the saved adapter from disk through Unsloth. This smoke test checks that the saved artifact can be opened independently of the in-memory training object.


In [ ]:
# Optional smoke test: reload the saved adapter from disk.
# This loads model weights and therefore remains disabled by default.

RUN_ADAPTER_RELOAD_TEST = False

if RUN_ADAPTER_RELOAD_TEST:
    if not os.path.isdir(ADAPTER_DIR):
        raise FileNotFoundError(f"Adapter directory not found: {ADAPTER_DIR}")

    reloaded_model, reloaded_tokenizer = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_DIR,
        max_seq_length=512,
        dtype=torch.float16,
        load_in_4bit=True
    )

    FastLanguageModel.for_inference(reloaded_model)
    reloaded_model.eval()

    print("Saved adapter reloaded successfully.")
else:
    print("Adapter reload test skipped.")


Optionally merge LoRA weights into the base model and export a standalone 16-bit checkpoint. This deployment-oriented step is resource-intensive and remains behind the export flag.


In [ ]:
# Optional merge/export step. Merging creates a standalone 16-bit model
# and can require substantially more RAM/VRAM and disk space than the adapter.

MERGED_DIR = "./outputs/supportmind/merged_16bit"

if MODEL_READY and TRAINING_COMPLETED and RUN_EXPORT:
    os.makedirs(MERGED_DIR, exist_ok=True)

    model.save_pretrained_merged(
        MERGED_DIR,
        tokenizer,
        save_method="merged_16bit"
    )

    print("Merged model exported to:", MERGED_DIR)
else:
    print("Merged export skipped. Complete training and set RUN_EXPORT=True.")


### 13. Final Inference

The final stage turns the trained SupportMind model into an inference interface. It accepts a new customer message, applies the Qwen chat template, generates tokens, removes the prompt tokens, parses the generated JSON, and validates the output schema before returning a Python dictionary.


Switch the loaded Unsloth model into inference mode. This prepares the model for generation and avoids training-oriented behavior during evaluation.


In [ ]:
from unsloth import FastLanguageModel
import torch

if MODEL_READY and TRAINING_COMPLETED:
    FastLanguageModel.for_inference(model)
    model.eval()
    print("SupportMind is ready for inference.")
else:
    print("Inference execution is gated until a trained model is available.")


Validate the incoming user message and build the system/user conversation expected by the model's chat template.


In [ ]:
def build_messages(user_message: str) -> list:
    if not isinstance(user_message, str) or not user_message.strip():
        raise ValueError("user_message must be a non-empty string.")

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message.strip()}
    ]


Implement the production-facing generation path: apply the chat template, tokenize and move inputs to the model device, generate deterministically, slice off prompt tokens, and decode only the new response.


In [ ]:
def generate_response(
    user_message: str,
    model,
    tokenizer,
    max_new_tokens: int = 256
) -> str:
    messages = build_messages(user_message)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(model.device)

    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    prompt_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][prompt_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


Parse the raw generated text as strict JSON and enforce the exact SupportMind schema before exposing it to downstream application code.


In [ ]:
import json

REQUIRED_FIELDS = {
    "intent",
    "response",
    "needs_clarification"
}


def parse_prediction(raw_output: str) -> dict:
    try:
        prediction = json.loads(raw_output)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON output: {exc}") from exc

    if not isinstance(prediction, dict):
        raise ValueError("The model output must be a JSON object.")

    if set(prediction) != REQUIRED_FIELDS:
        raise ValueError("The model output does not match the required schema.")

    if not isinstance(prediction["intent"], str) or prediction["intent"] not in INTENTS:
        raise ValueError("The model returned an unsupported intent.")

    if not isinstance(prediction["response"], str) or not prediction["response"].strip():
        raise ValueError("The response must be a non-empty string.")

    if not isinstance(prediction["needs_clarification"], bool):
        raise ValueError("needs_clarification must be a boolean.")

    return prediction


#### Unit Tests for Output Validation

These tests validate the JSON parser and schema checks without loading the language model. They cover a valid prediction, malformed JSON, an unsupported intent, and incorrect field types. This makes it possible to test an important part of the inference pipeline even when model weights are not downloaded.


Run model-independent unit tests for the JSON parser. These tests can execute even when the base model has not been downloaded.


In [ ]:
import unittest

class PredictionParserTests(unittest.TestCase):
    def test_valid_prediction(self):
        raw = json.dumps({
            "intent": "return_request",
            "response": "Please provide your order ID.",
            "needs_clarification": True
        })
        parsed = parse_prediction(raw)
        self.assertEqual(parsed["intent"], "return_request")

    def test_invalid_json(self):
        with self.assertRaises(ValueError):
            parse_prediction("{not valid json}")

    def test_unknown_intent(self):
        raw = json.dumps({
            "intent": "unknown_intent",
            "response": "Hello.",
            "needs_clarification": False
        })
        with self.assertRaises(ValueError):
            parse_prediction(raw)

    def test_wrong_boolean_type(self):
        raw = json.dumps({
            "intent": "other",
            "response": "Hello.",
            "needs_clarification": "false"
        })
        with self.assertRaises(ValueError):
            parse_prediction(raw)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(PredictionParserTests)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()


Combine generation and schema validation into one callable interface that accepts a customer message and returns a validated Python dictionary.


In [ ]:
def predict_supportmind(
    user_message: str,
    model,
    tokenizer
) -> dict:
    raw_output = generate_response(
        user_message=user_message,
        model=model,
        tokenizer=tokenizer
    )

    return parse_prediction(raw_output)


Prepare a small set of representative customer messages covering order status, returns, billing, product questions, and greetings for a final inference smoke test.


In [ ]:
test_messages = [
    "Where is my order? My order ID is 78921.",
    "I received my shoes, but they are damaged. I want to return them.",
    "The payment was deducted from my account, but my order was not confirmed.",
    "Can you tell me more about your products?",
    "Hello!"
]


Run the final SupportMind interface on representative messages and display validated JSON results. Live generation is skipped unless the model has been loaded and trained.


In [ ]:
if MODEL_READY and TRAINING_COMPLETED:
    for message in test_messages:
        print("=" * 70)
        print("USER:", message)

        try:
            prediction = predict_supportmind(
                user_message=message,
                model=model,
                tokenizer=tokenizer
            )

            print("\nSUPPORTMIND:")
            print(json.dumps(
                prediction,
                indent=4,
                ensure_ascii=False
            ))

        except ValueError as exc:
            print("\nPREDICTION ERROR:", exc)
else:
    print(
        "Live inference was not executed because model download/training "
        "is disabled. The complete inference functions are defined above."
    )


## Project Completion Checklist

The notebook now contains the complete learning workflow: environment setup, model and tokenizer inspection, dataset engineering, leakage-safe splitting, chat-template-aware tokenization, precision and quantization analysis, LoRA configuration, TRL SFT training, training diagnostics, structured evaluation, human response-quality review, adapter persistence, optional merged export, and final inference.

To run the model-dependent parts, first confirm the runtime and package versions, then enable `RUN_MODEL_DOWNLOADS`, `RUN_TRAINING`, `RUN_EVALUATION`, and `RUN_EXPORT` as needed. The default settings intentionally avoid downloading Qwen3-4B weights or starting GPU training.

The current dataset is synthetic and very small. Treat all measured scores as educational diagnostics, not evidence of production readiness.
